# 02 · 临近预报基线（Persistence / PySTEPS* / Conv baseline）

**目的**：在与扩散模型相同的数据与划分上，建立可复现的弱基线与中等基线，输出 MAE 与 B13@240K CSI，供论文对比。

**说明**：
* 主评分使用 **B13**；PySTEPS 外推仅在 **B13 场** 上运行（亮温非降水率，光流外推在形态上仍具参照意义）。
* 若未安装 ``pysteps``，第 4 节会跳过并提示安装。


## 步骤 0：环境检查

1. ``conda activate xn``，且 ``import torch`` 成功。
2. 已设置 ``XN_TRAIN_DIR / XN_VAL_DIR / XN_TEST_DIR``（与 01 审计相同）。
3. 在仓库根目录启动 Jupyter；下方 cell 会把仓库根加入 ``sys.path``。


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data.h8_dataset import H8Dataset, load_blacklist
from src.data.normalizers import norm_to_kelvin, B13_INDEX
from src.metrics.csi import csi_at_threshold_k
from src.models.baselines.convlstm_nowcast import ConcatConvNowcast

print("REPO_ROOT =", REPO_ROOT)
print("torch", torch.__version__, "cuda?", torch.cuda.is_available())


## 步骤 1：路径与数据集


In [ ]:
TRAIN_DIR = os.environ.get("XN_TRAIN_DIR", "/share/home/sera_hujun/train_data_v7_unbiased_501")
VAL_DIR = os.environ.get("XN_VAL_DIR", "/share/home/sera_hujun/val_data_v7_unbiased_501")
TEST_DIR = os.environ.get("XN_TEST_DIR", "/share/home/sera_hujun/test_data_v7_unbiased_501")
BLACKLIST = os.environ.get("XN_BLACKLIST", str(REPO_ROOT / "problematic_checkpoints.csv"))

for name, p in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    print(name, p, "exists=", Path(p).exists())

bl = load_blacklist(BLACKLIST)
ds_test = H8Dataset(TEST_DIR, mode="split", blacklist=bl)
print("test samples:", len(ds_test))


## 步骤 2：评估函数（B13 MAE + 逐帧 CSI@240K 再平均）

**本 cell 只定义函数，运行后应看到一行 `[步骤 2] 完成`。** 没有表格是正常的。


In [ ]:
THRESH_K = 240.0


def b13_kelvin(x_cthw: torch.Tensor) -> torch.Tensor:
    b13n = x_cthw[B13_INDEX]
    return norm_to_kelvin(b13n, "B13")


def mae_b13_k(pred: torch.Tensor, true: torch.Tensor) -> float:
    pk = b13_kelvin(pred).numpy()
    tk = b13_kelvin(true).numpy()
    return float(np.mean(np.abs(pk - tk)))


def mean_csi_b13_240(pred: torch.Tensor, true: torch.Tensor) -> float:
    pk = b13_kelvin(pred).numpy()
    tk = b13_kelvin(true).numpy()
    T = pk.shape[0]
    csis = [csi_at_threshold_k(pk[t], tk[t], THRESH_K)["CSI"] for t in range(T)]
    return float(np.mean(csis))


def evaluate_forecast(pred_future: torch.Tensor, true_future: torch.Tensor) -> dict:
    return {
        "MAE_B13_K": mae_b13_k(pred_future, true_future),
        "mean_CSI_B13_240K": mean_csi_b13_240(pred_future, true_future),
    }

print("[步骤 2] 完成：已定义 evaluate_forecast / MAE_B13_K / mean_CSI_B13_240K")
print("       本 cell 无表格输出是正常的；结果在步骤 3、5 打印。")


## 步骤 3：基线 A — Persistence（末帧复制 12 步）


In [ ]:
@torch.no_grad()
def baseline_persistence(past: torch.Tensor) -> torch.Tensor:
    c, tp, h, w = past.shape
    return past[:, -1:, :, :].expand(-1, 12, -1, -1).clone()


N_EVAL = min(200, len(ds_test))
rows = []
for i in range(N_EVAL):
    batch = ds_test[i]
    past, fut = batch["past"], batch["future"]
    pred = baseline_persistence(past)
    m = evaluate_forecast(pred, fut)
    m["i"] = i
    rows.append(m)

df_p = pd.DataFrame(rows)
print(df_p.describe().T)


## 步骤 4：基线 B — PySTEPS（可选）

若导入失败: ``pip install pysteps`` 后重启 kernel。

**注意**：不同 ``pysteps`` 版本的 ``dense_lucaskanade`` 输入维度可能不同；若本节报错，请根据你环境内的 API 调整 ``R`` 的形状，或暂时跳过本节。


In [ ]:
PYSTEPS_OK = False
try:
    from pysteps.motion.lucaskanade import dense_lucaskanade
    from pysteps.extrapolation.semilagrangian import extrapolate

    PYSTEPS_OK = True
    print("pysteps: OK")
except Exception as e:
    print("pysteps: SKIP ->", e)


@torch.no_grad()
def baseline_pysteps_b13(past: torch.Tensor, n_lead: int = 12) -> torch.Tensor:
    c, tp, h, w = past.shape
    out = past[:, -1:, :, :].expand(-1, n_lead, -1, -1).clone()
    if not PYSTEPS_OK:
        return out
    b13n = past[B13_INDEX].numpy().astype(np.float32)
    R = (-b13n).copy()
    # 常见 API: 输入 (T,ny,nx) 或 (1,T,ny,nx)；若报错请查阅本地 pysteps 文档改此处
    V = dense_lucaskanade(R[-2:])
    last = R[-1]
    seq = extrapolate(last, V, n_lead)
    out[B13_INDEX] = torch.from_numpy((-seq).astype(np.float32))
    return out


if PYSTEPS_OK:
    rows2 = []
    for i in range(N_EVAL):
        batch = ds_test[i]
        past, fut = batch["past"], batch["future"]
        pred = baseline_pysteps_b13(past)
        m = evaluate_forecast(pred, fut)
        m["i"] = i
        rows2.append(m)
    df_s = pd.DataFrame(rows2)
    print(df_s.describe().T)
else:
    print("跳过 PySTEPS 评估")


## 步骤 5：基线 C — ConcatConv（加速版：256 裁剪 + DataLoader）

**为何 GPU 利用率只有 ~5%？** 旧写法每次 `ds[i]` 从磁盘读 **501×501** 单条样本，CPU/I/O 是瓶颈，GPU 大部分时间在等数据。

本 cell 改为：**256×256 随机裁剪** + **batch=8** + **4 进程读盘**，A100 上 GPU 利用率会明显升高，约 **5–15 分钟** 跑完 300 step。

> 步骤 3 Persistence 若在全图 501 上跑的，数值与本次 256 不完全可比；若要严格对比，可用同样 `CROP=256` 重跑步骤 3（可选）。


In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from src.data.transforms import CropTransform

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CROP = 256          # 与 Stage-A VAE 训练一致，显著加速
SUBSET = 512
MAX_STEPS = 300
BATCH_SIZE = 8
NUM_WORKERS = 4
LR = 3e-4

print("[步骤 5] ConcatConv（加速版）")
print(f"  DEVICE={DEVICE}, CROP={CROP}, BATCH={BATCH_SIZE}, workers={NUM_WORKERS}, steps={MAX_STEPS}")

ds_tr = H8Dataset(TRAIN_DIR, mode="split", blacklist=bl)
g = torch.Generator().manual_seed(2025)
perm = torch.randperm(len(ds_tr), generator=g)[:SUBSET]
subset_paths = [ds_tr.metas[int(i)].path for i in perm]

crop_train = CropTransform(CROP, mode="random")
crop_eval = CropTransform(CROP, mode="center")
ds_small = H8Dataset(TRAIN_DIR, mode="split", blacklist=bl, files=subset_paths, crop=crop_train)
ds_test_crop = H8Dataset(TEST_DIR, mode="split", blacklist=bl, crop=crop_eval)

train_loader = DataLoader(
    ds_small,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

model_c = ConcatConvNowcast().to(DEVICE)
opt = torch.optim.AdamW(model_c.parameters(), lr=LR)
loss_fn = nn.L1Loss()
model_c.train()

step = 0
pbar = tqdm(total=MAX_STEPS, desc="ConcatConv train")
while step < MAX_STEPS:
    for batch in train_loader:
        past_b = batch["past"].to(DEVICE, non_blocking=True)
        fut_b = batch["future"].to(DEVICE, non_blocking=True)
        pred = model_c(past_b)
        loss = loss_fn(pred, fut_b)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        step += past_b.size(0)
        pbar.update(past_b.size(0))
        pbar.set_postfix(loss=float(loss.item()), bs=past_b.size(0))
        if step >= MAX_STEPS:
            break
pbar.close()
print(f"[步骤 5] 训练结束 (约 {step} 次参数更新)，开始评估 ...")

model_c.eval()
rows3 = []
with torch.no_grad():
    for i in tqdm(range(N_EVAL), desc="ConcatConv eval"):
        b = ds_test_crop[i]
        p = b["past"].unsqueeze(0).to(DEVICE)
        f = b["future"]
        pr = model_c(p).cpu()[0]
        rows3.append(evaluate_forecast(pr, f))

df_cc = pd.DataFrame(rows3)
print("\n[步骤 5] ConcatConv 测试集均值 (N_EVAL=%d, CROP=%d):" % (N_EVAL, CROP))
print(df_cc.mean())
print(df_cc.describe().T)


## 步骤 6：导出 CSV

**必须先完成步骤 3 和步骤 5。** 运行后应逐行打印已写入的 csv 文件名。


In [ ]:
print("[步骤 6] 导出 CSV ...")

missing = []
for name in ("df_p", "rows3", "N_EVAL"):
    if name not in globals():
        missing.append(name)
if missing:
    raise NameError(
        f"缺少变量 {missing}。请先运行：步骤 3（得到 df_p）、步骤 5（得到 rows3），"
        "且步骤 3 之前需运行步骤 1（得到 N_EVAL）。"
    )

out_dir = REPO_ROOT / "reports" / "baselines"
out_dir.mkdir(parents=True, exist_ok=True)

df_p.to_csv(out_dir / "persistence_test_subset.csv", index=False)
print("  已写:", out_dir / "persistence_test_subset.csv")

if "PYSTEPS_OK" in globals() and PYSTEPS_OK and "df_s" in globals():
    df_s.to_csv(out_dir / "pysteps_b13_test_subset.csv", index=False)
    print("  已写:", out_dir / "pysteps_b13_test_subset.csv")
else:
    print("  跳过 pysteps CSV（未运行或 PYSTEPS_OK=False）")

pd.DataFrame(rows3).to_csv(out_dir / "concatconv_test_subset.csv", index=False)
print("  已写:", out_dir / "concatconv_test_subset.csv")

print("\n[步骤 6] 完成。目录内容:")
for f in sorted(out_dir.glob("*.csv")):
    print(" ", f.name, f.stat().st_size, "bytes")


## 下一步

1. Stage-A VAE: ``python scripts/train_stvae.py``（见 README）。
2. VAE 重建 B13@240K CSI 建议 >= 0.95 再进入扩散主干。
